# Worker C: Knapsack alpha=2.0

Runs the alpha-fair knapsack experiment at alpha=2.0 across all unfairness levels.

**Grid:** 7 methods x 5 seeds x 3 unfairness levels = 105 runs

Checkpointed: re-run cells to resume from where you left off.

In [ ]:
import os, sys

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ----- Path setup -----
DRIVE_ROOT = '/content/drive/MyDrive/DecisionFocusedMTL'
MOSEK_LIC_PATH = f'{DRIVE_ROOT}/data/mosek.lic'

if os.path.exists(MOSEK_LIC_PATH):
    os.environ['MOSEKLM_LICENSE_FILE'] = MOSEK_LIC_PATH
    print(f'MOSEK license set: {MOSEK_LIC_PATH}')
else:
    print(f'Warning: MOSEK license not found at {MOSEK_LIC_PATH}')

try:
    import mosek
    print(f'MOSEK installed: {mosek.Env().getversion()}')
except ImportError:
    print('Installing MOSEK...')
    os.system('pip install mosek -q')
    print('MOSEK installed.')

for p in [DRIVE_ROOT, os.path.join(DRIVE_ROOT, 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

%cd {DRIVE_ROOT}

# Patch solver chain: MOSEK > CLARABEL > SCS
import fair_dfl.tasks.md_knapsack as _kn
import cvxpy as cp
_kn._SOLVER_CHAIN = [
    (cp.MOSEK,    {}),
    (cp.CLARABEL, {}),
    (cp.SCS,      {'eps': 1e-6, 'max_iters': 10000}),
]

# Results directory
KN_RESULTS = os.path.join(DRIVE_ROOT, 'results', 'final', 'knapsack')
os.makedirs(KN_RESULTS, exist_ok=True)

from experiments.colab_runner import *
print(f'Ready. Results: {KN_RESULTS}')

## Configuration

Edit this cell to change any parameter. Re-run the cell, then re-run the experiment cell below.

In [ ]:
# ===== TASK PARAMETERS =====
TASK_OVERRIDES = {
    'n_items': 7,              # Number of items in knapsack
    'n_budget_dims': 1,        # Budget dimensions (rows of A matrix). 1 = classic knapsack
    'n_features': 5,           # Input feature dimension
    'budget_tightness': 0.3,   # b = tightness * A.sum(). Lower = tighter = more competition
    'poly_degree': 2,          # Polynomial feature mapping degree
    'decision_mode': 'group',  # Two-level group alpha-fairness
    'n_samples_train': 400,    # Training samples
    'n_samples_val': 80,       # Validation samples (for monitoring, early stopping)
    'n_samples_test': 200,     # Test samples
}

# ===== UNFAIRNESS LEVELS =====
# group_bias / noise_std_hi > 1.5 ensures learnable signal
UF_CONFIGS = {
    'mild':   {'group_bias': 0.2, 'noise_std_lo': 0.05, 'noise_std_hi': 0.10, 'group_ratio': 0.5},
    'medium': {'group_bias': 0.4, 'noise_std_lo': 0.05, 'noise_std_hi': 0.20, 'group_ratio': 0.6},
    'high':   {'group_bias': 0.6, 'noise_std_lo': 0.05, 'noise_std_hi': 0.30, 'group_ratio': 0.75},
}

# ===== TRAINING PARAMETERS =====
TRAIN_OVERRIDES = {
    # Optimizer
    'optimizer': 'adam',            # 'sgd' or 'adam'
    'lr': 0.001,                   # Learning rate
    'weight_decay': 1e-4,          # L2 regularization
    # Decision gradient
    'decision_grad_backend': 'spsa',  # 'spsa' or 'finite_diff'
    'batch_size': 32,              # Same batch for all methods (no confound)
    # Model architecture
    'init_mode': 'best_practice',  # Kaiming He initialization
    'dropout': 0.1,                # Dropout rate
    'hidden_dim': 64,              # MLP hidden width
    'n_layers': 2,                 # MLP depth
}

# ===== EXPERIMENT CONTROL =====
STEPS = 70                 # Gradient updates per lambda stage
OVERWRITE = False          # True = re-run completed experiments

# ===== PRINT SUMMARY =====
print('='*60)
print('EXPERIMENT CONFIGURATION')
print('='*60)
print(f'Alpha: 2.0')
print(f'Steps per lambda: {STEPS}')
print(f'Overwrite: {OVERWRITE}')
print(f'\nTask: {TASK_OVERRIDES}')
print(f'\nUnfairness levels:')
for k, v in UF_CONFIGS.items():
    snr = v['group_bias'] / v['noise_std_hi']
    print(f'  {k}: bias={v["group_bias"]}, noise_hi={v["noise_std_hi"]}, ratio={v["group_ratio"]}, SNR~{snr:.1f}')
print(f'\nTrain overrides: {TRAIN_OVERRIDES}')
print(f'Results dir: {KN_RESULTS}')
print('='*60)

## Run All Unfairness Levels

In [ ]:
results = run_knapsack_slice(
    alphas=[2.0],
    unfairness_levels=list(UF_CONFIGS.keys()),
    results_dir=KN_RESULTS,
    steps=STEPS,
    task_overrides=TASK_OVERRIDES,
    unfairness_configs=UF_CONFIGS,
    train_overrides=TRAIN_OVERRIDES,
    overwrite=OVERWRITE,
)

In [ ]:
show_progress(KN_RESULTS, 'Knapsack alpha=2.0 — ALL LEVELS')

if not results.empty:
    summary = results.groupby(['method_label', 'lambda', 'unfairness_level']).agg({
        'test_regret_normalized': ['mean', 'std'],
        'test_fairness': ['mean', 'std'],
        'test_pred_mse': ['mean', 'std'],
    }).round(4)
    print('\n--- Results Summary (mean +/- std over seeds) ---')
    print(summary.to_string())

Worker C complete. Run the Results notebook to analyze.